## only works with 3.10+

In [ ]:
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator

In [ ]:
triplets_df_path = "../data/TRIPLETS_ALL.csv"
df = pd.read_csv(triplets_df_path)
df_sub = df[["entity1", "rel_type", "entity2"]]
df_sub = df_sub.drop_duplicates(ignore_index=True)

In [ ]:
# create full triples factory 
tf = TriplesFactory.from_labeled_triples(
    df_sub.values,
    create_inverse_triples=True,
)

# use PyKEEN split
train_tf, val_tf, test_tf = tf.split(
    ratios=(0.7, 0.15, 0.15),
    random_state=11,
)

In [ ]:
print(train_tf.num_entities)
print(train_tf.num_relations)
print(train_tf.num_triples)

In [ ]:
print(train_tf.num_triples + val_tf.num_triples + test_tf.num_triples)

In [ ]:
import torch

rel_id = train_tf.relation_to_id["invests_in"]
count = (train_tf.mapped_triples[:,1] == rel_id).sum()
print(count)

In [ ]:
result = pipeline(
    training=train_tf,
    validation=val_tf,
    testing=test_tf,
    model="ComplEx",
    model_kwargs=dict(
        embedding_dim=128,
    ),
    training_kwargs=dict(
        num_epochs=150,
        batch_size=1024,
    ),
    optimizer="adam",
    optimizer_kwargs=dict(
        lr=1e-3,
    ),
    negative_sampler="basic",
    negative_sampler_kwargs=dict(
        num_negs_per_pos=25,
    ),
    evaluator_kwargs=dict(filtered=True),
    random_seed=11,
)

In [ ]:
metrics = result.metric_results.to_dict()
print(metrics)